# TVBx Minimal BG Chain Experiment

This notebook runs a minimal basal-ganglia (BG) co-simulation chain using the new `tvbx_multiscale` scaffolding.
It uses mock TVB and mock SNN adapters to validate the loop end-to-end.

In [ ]:
import pathlib
import sys

cwd = pathlib.Path.cwd().resolve()
candidates = [cwd / "src", cwd.parent / "src"]
src_path = None
for candidate in candidates:
    if candidate.exists():
        src_path = candidate
        break

if src_path is None:
    raise RuntimeError(f"Could not locate src/ from cwd={cwd}")

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Using src path: {src_path}")

In [ ]:
import numpy as np

from tvbx_multiscale.core import CoSimulationOrchestrator
from tvbx_multiscale.adapters.nest import NESTMockAdapter
from tvbx_multiscale.adapters.tvboptim import TVBOptimMockAdapter
from tvbx_multiscale.domain.basal_ganglia import BasalGangliaNodeGroups
from tvbx_multiscale.transforms import RateSNNToTVB, RateTVBToSNN

In [ ]:
bg = BasalGangliaNodeGroups()

node_labels = {
    0: "iGPe_L",
    1: "iGPe_R",
    2: "iGPi_L",
    3: "iGPi_R",
    4: "eSTN_L",
    5: "eSTN_R",
    6: "iSTR_L",
    7: "iSTR_R",
    8: "eTH_L",
    9: "eTH_R",
}

for idx in range(10):
    print(idx, node_labels[idx])

In [ ]:
steps = 80
n_nodes = 10
tvb_dt_ms = 1.0

tvb = TVBOptimMockAdapter(n_nodes=n_nodes, dt_ms=tvb_dt_ms, baseline_rate_hz=5.0)
snn = NESTMockAdapter(n_nodes=n_nodes, dt_ms=tvb_dt_ms, saturation_hz=120.0)

tvb_to_snn = RateTVBToSNN(interface_weight=1.0, min_rate_hz=0.0)
snn_to_tvb = RateSNNToTVB(spikes_to_rate_scale=1000.0 / tvb_dt_ms)

orchestrator = CoSimulationOrchestrator(
    tvb_adapter=tvb,
    snn_adapter=snn,
    tvb_to_snn=tvb_to_snn,
    snn_to_tvb=snn_to_tvb,
)

orchestrator.initialize()
traces = orchestrator.run(steps=steps)

print(f"steps={len(traces)}")
print(f"last_time_ms={traces[-1].tvb_output.time_ms:.3f}")
print(f"last_tvb_mean_rate_hz={traces[-1].tvb_output.rate_by_node.mean():.3f}")
print(f"last_snn_mean_ratio={traces[-1].snn_output.population_mean_spikes_number_by_node.mean():.5f}")

In [ ]:
times = np.array([t.tvb_output.time_ms for t in traces], dtype=np.float64)
tvb_mean_rate = np.array([t.tvb_output.rate_by_node.mean() for t in traces], dtype=np.float64)
snn_mean_ratio = np.array(
    [t.snn_output.population_mean_spikes_number_by_node.mean() for t in traces],
    dtype=np.float64,
)
feedback_mean_rate = np.array([t.tvb_feedback.value_by_node.mean() for t in traces], dtype=np.float64)

print("first 5 tvb mean rates:", np.round(tvb_mean_rate[:5], 4))
print("last 5 tvb mean rates:", np.round(tvb_mean_rate[-5:], 4))
print("first 5 snn mean ratios:", np.round(snn_mean_ratio[:5], 5))
print("last 5 snn mean ratios:", np.round(snn_mean_ratio[-5:], 5))

In [ ]:
try:
    import matplotlib.pyplot as plt

    plt.figure(figsize=(10, 4))
    plt.plot(times, tvb_mean_rate, label="TVB mean rate (Hz)", linewidth=2)
    plt.plot(times, feedback_mean_rate, label="Feedback mean rate (Hz)", linestyle="--", linewidth=2)
    plt.xlabel("Time (ms)")
    plt.ylabel("Rate (Hz)")
    plt.title("BG Minimal Chain: TVB and Feedback")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(times, snn_mean_ratio, label="SNN mean spike ratio", color="tab:orange", linewidth=2)
    plt.xlabel("Time (ms)")
    plt.ylabel("Spike ratio")
    plt.title("BG Minimal Chain: SNN Activity")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()
except Exception as exc:
    print("matplotlib not available:", exc)

In [ ]:
group_index = {
    "iGPe": bg.igpe_nodes,
    "iGPi": bg.igpi_nodes,
    "eSTN": bg.estn_nodes,
    "iSTR": bg.istr_nodes,
    "eTH": bg.eth_nodes,
}

last_tvb = traces[-1].tvb_output.rate_by_node
last_snn = traces[-1].snn_output.population_mean_spikes_number_by_node

for group_name, group_nodes in group_index.items():
    tvb_group_mean = float(np.mean(last_tvb[list(group_nodes)]))
    snn_group_mean = float(np.mean(last_snn[list(group_nodes)]))
    print(f"{group_name:>4} | TVB mean Hz={tvb_group_mean:8.3f} | SNN mean ratio={snn_group_mean:8.5f}")

In [ ]:
def check_runtime_adapters():
    print("Runtime adapter availability check")
    print("-" * 36)

    try:
        import jax  # noqa: F401
        import tvboptim  # noqa: F401
        from tvbx_multiscale.adapters.tvboptim import TVBOptimRuntimeAdapter  # noqa: F401
        print("TVBOptimRuntimeAdapter: ready (jax + tvboptim importable)")
    except Exception as exc:
        print(f"TVBOptimRuntimeAdapter: unavailable ({exc})")

    try:
        import nest  # noqa: F401
        from tvbx_multiscale.adapters.nest import NESTLegacyRuntimeAdapter  # noqa: F401
        print("NESTLegacyRuntimeAdapter: ready (nest importable)")
    except Exception as exc:
        print(f"NESTLegacyRuntimeAdapter: unavailable ({exc})")


check_runtime_adapters()